In [ ]:
import pandas as pd
from main import SatCLIPLightningModule
import location_encoder as LE
from temporal_encoding import Fourier, Direct
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import numpy as np

from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import io
from PIL import Image

from utils import *
import torch.nn.functional as F

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create the spatiotemporal train-test splits

In [ ]:
# Load the GHCN dataset
ghcn_df = pd.read_csv('../data/ghcn_data/ghcn_2021_2026_temperature.csv')
print(len(ghcn_df))
ghcn_df.head()

In [ ]:
# Filter GHCN to quality controlled data and limit to TMAX (maximum temperature) observations
ghcn_qc_temp_df = ghcn_df[(ghcn_df["Element"]=="TMAX") & (ghcn_df["QFlag"].isna())].copy()
ghcn_qc_temp_df['Date'] = pd.to_datetime(ghcn_qc_temp_df['Date'], format="%Y%m%d")
ghcn_qc_temp_df.loc[:, 'DayOfYear'] = (ghcn_qc_temp_df['Date'].dt.dayofyear - 1) / 364.0
ghcn_qc_temp_df.head()

In [ ]:
# Plot the temperatures on a map
PLOT_TEMPS = False

if PLOT_TEMPS:
    fig, ax = plt.subplots(figsize=(12, 6))
    m = Basemap(projection='cyl', resolution='c', ax=ax)
    m.drawcoastlines()
    sc = ax.scatter(ghcn_qc_temp_df['Longitude'].values, ghcn_qc_temp_df['Latitude'].values, c=ghcn_qc_temp_df['Value'].values/10, cmap='coolwarm', marker='o', alpha=0.5)
    fig.colorbar(sc, label='Temperature (°C)', ax=ax)

In [ ]:
# Plot histogram of observations over time for all stations
plt.figure(figsize=(12, 6))
plt.hist(ghcn_qc_temp_df['Date'], bins=30, edgecolor='black')
plt.xlabel('Date')
plt.ylabel('Frequency')
plt.title('Distribution of Observations Over Time')
plt.show()

In [ ]:
# Plot the distribution of temperatures over time for a randomly selected station
import random
random_station = random.choice(ghcn_qc_temp_df['ID'].unique())
station_df = ghcn_qc_temp_df[ghcn_qc_temp_df['ID']== random_station]
plt.figure(figsize=(12, 6))
plt.scatter(station_df['Date'], station_df['Value']/10, marker='o', linestyle='-', label=f'Station: {random_station}')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Over Time for Station {random_station}')

# Create Subsampled Datasets for Quick Testing

In [ ]:
# Extract the relevant columns for spatial evaluation (ALWAYS RUN THIS BEFORE SUBSAMPLING)
temp_data = torch.tensor(ghcn_qc_temp_df[['Longitude', 'Latitude', 'DayOfYear', 'Value']].values, dtype=torch.float32)
dates = ghcn_qc_temp_df['Date']
ids = ghcn_qc_temp_df['ID']
all_station_ids = ids.unique()

print("Number of observations:", temp_data.shape[0])
print("Number of unique stations:", len(all_station_ids))

In [ ]:
# Subsample the data and prepare it for pytorch
SUBSAMPLE_BY_STATION = False

if SUBSAMPLE_BY_STATION:
    # Subsample by stations
    NUM_STATIONS = 2000
    sampled_station_ids = np.random.choice(all_station_ids, size=NUM_STATIONS, replace=False)
    temp_data = temp_data[ids.isin(sampled_station_ids).to_numpy(), :]
    dates = dates[ids.isin(sampled_station_ids).to_numpy()]
    ids = ids[ids.isin(sampled_station_ids).to_numpy()]
    print("Number of sampled stations:", len(sampled_station_ids))
    print("Number of sampled observations:", temp_data.shape[0])
else:
    # Subsample by samples
    NUM_SAMPLES = 500_000
    sampled_indices = np.random.choice(temp_data.shape[0], size=NUM_SAMPLES, replace=False)
    temp_data = temp_data[sampled_indices, :]
    dates = dates.iloc[sampled_indices]
    ids = ids.iloc[sampled_indices]
    print("Number of sampled observations:", temp_data.shape[0])

In [ ]:
# Setup cross validation splits (either spatial or temporal)
SPATIAL_CV = False

if SPATIAL_CV:
    # Create the spatial checkerboard splits for the GHCN dataset
    delta_degree = 32
    splits = checkerboard_splits(temp_data[:, :2], torch.tensor([delta_degree, delta_degree])).flatten()

else:
    # Create temporal splits for the GHCN dataset
    delta_day = 90
    # convert dates into days since the first date
    days_since_first = torch.tensor((dates - dates.min()).dt.days.to_numpy())
    splits = temporal_splits(days_since_first, delta_day)

In [ ]:
# Plot the splits on a map
fig, ax = plt.subplots(figsize=(12, 6))
m = Basemap(projection='cyl', resolution='c', ax=ax)
m.drawcoastlines()
# Parallels (Latitudes) range from -90 to 90
m.drawparallels(np.arange(-90, 90, 30), labels=[True, True, False, False])

# Meridians (Longitudes) range from 0 to 360 (or -180 to 180 depending on projection)
m.drawmeridians(np.arange(0, 360, 30), labels=[False, False, True, True])

sc = ax.scatter(temp_data[:, 0], temp_data[:, 1], c=splits, cmap='coolwarm', marker='o', alpha=0.5)
fig.colorbar(sc, label='Split', ax=ax)

In [ ]:
# Plot GIF of the dataset over time
PLOT_GIF = False

frames = []

days_win = 30

# sample a subset of dates every days_win days to reduce the number of frames in the GIF
subset_dates = dates.sort_values().unique()
subset_dates = subset_dates[::days_win]

for date in tqdm(subset_dates):
    if not PLOT_GIF:
        break
    
    # Find images within a window of days
    temp_window = temp_data[(dates == date).to_numpy(), :]
    dates_window = dates[(dates == date)]

    # cs = np.array([np.array(x)/255.0 for x in df_window['avg_color'].values])
    # cs = np.minimum(cs, 1.0)

    # Plot the average colors of the images in this window on the map
    fig, ax = plt.subplots(1,1, figsize=(12, 6))

    m = Basemap(projection='cyl', resolution='c', ax=ax)
    m.drawcoastlines()
    m.drawparallels(np.arange(-90, 90, 30), labels=[True, True, False, False])
    m.drawmeridians(np.arange(0, 360, 30), labels=[False, False, True, True])

    sc = ax.scatter(temp_window[:,0], temp_window[:,1], c=temp_window[:,3]/10, cmap='coolwarm', marker='o', alpha=0.5)
    ax.set_title(f"Date: {date.strftime('%Y-%m-%d')}")
    ax.legend()

    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    frames.append(Image.open(buf))
    plt.close(fig)

    if len(frames) > 30:
        break

# Save as GIF
if PLOT_GIF:
    frames[0].save('results/ghcn_temporal_evolution.gif', format='GIF', append_images=frames[1:], save_all=True, duration=500, loop=0)

In [ ]:
# Plot data for a single station over time from the subset
rand_row = random.choice(range(len(temp_data)))
rand_station_id = ids.iloc[rand_row]
station_data = temp_data[(ids == rand_station_id).to_numpy(), :]
station_dates = dates[(ids == rand_station_id).to_numpy()]
station_splits = splits[(ids == rand_station_id).to_numpy()]

# Plot the temperature over time for this station
split_id = 0
plt.figure(figsize=(12, 6))
plt.scatter(station_dates, station_data[:, 3]/10, marker='o', linestyle='-', label=f'All data from station')
plt.scatter(station_dates[station_splits.numpy() == split_id], station_data[station_splits.numpy() == split_id, 3]/10, marker='o', color="red", linestyle='-', label=f'Split {split_id}')

plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Over Time for Station ID {rand_station_id}')
plt.legend()

# Perform fine-tuning for a location encoder

In [ ]:
# Free up GPU memory by emptying the cache
torch.cuda.empty_cache()

In [ ]:
# Setup initial datasets
train_split = 0

train = temp_data[splits == train_split, :]
test = temp_data[splits != train_split, :]

# Get a sample of the train and test splits
train_sample = train
test_sample = test
# train_sample = train[np.random.choice(train.shape[0], size=100000, replace=False)]
# test_sample = test[np.random.choice(test.shape[0], size=100000, replace=False)]

# Sample the embeddings for the train split
train_coords = train_sample[:, :3].detach().clone()
test_coords = test_sample[:, :3].detach().clone()

In [ ]:
EVAL_TYPE = "mlp" # Can be linear or mlp
print(f"Train coords shape: {train_coords.shape}")
print(f"Test coords shape: {test_coords.shape}")

In [ ]:
# Load a pre-trained spatio-temporal SatCLIP encoder
# ckpt_path = '/home/leca5365/Documents/satclip/satclip/satclip_temporal_logs/satclip-s2-temporal-55k-temporalsplits-fixednormalizeddayofyear/satclip-fixed-normalized-day-of-year/checkpoints/last.ckpt'
# ckpt_path = '/home/leca5365/Documents/satclip/satclip/satclip_temporal_logs/satclip-s2-satcliploss-100k/satclip-s2-satcliploss-100k/checkpoints/last-v2.ckpt'
# ckpt_path = '/home/leca5365/Documents/satclip/satclip/satclip_temporal_logs/satclip-s2-softloss-100k/satclip-s2-softloss-100k/checkpoints/last.ckpt'
ckpt_path = '/home/leca5365/Documents/satclip/satclip/satclip_temporal_logs/satclip-s2-satcliploss-1M/satclip-s2-satcliploss-1M/checkpoints/last-v1.ckpt'

lightning_model = SatCLIPLightningModule.load_from_checkpoint(ckpt_path)

lightning_model.eval()
spatiotemporal_enc = lightning_model.model.location
visual_enc = lightning_model.model.visual

# Get the embeddings for the GHCN dataset using the pre-trained spatio-temporal SatCLIP encoder
with torch.no_grad():
    train_embeddings = spatiotemporal_enc(train_coords.double().to(device)).detach()
    test_embeddings = spatiotemporal_enc(test_coords.double().to(device)).detach()

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, dim_hidden, num_layers, out_dims):
        super(MLP, self).__init__()

        layers = []
        layers += [nn.Linear(input_dim, dim_hidden, bias=True), nn.ReLU()] # Input layer
        layers += [nn.Linear(dim_hidden, dim_hidden, bias=True), nn.ReLU()] * num_layers # Hidden layers
        layers += [nn.Linear(dim_hidden, out_dims, bias=True)] # Output layer

        self.features = nn.Sequential(*layers)

    def forward(self, x):
        return self.features(x)

In [ ]:
# Create a DataLoader for the train embeddings and values
train_dataset = torch.utils.data.TensorDataset(train_embeddings, train_sample[:, 3].detach().clone().unsqueeze(1).double())
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8096, shuffle=True)

# Test dataloader
test_dataset = torch.utils.data.TensorDataset(test_embeddings, test_sample[:, 3].detach().clone().unsqueeze(1).double())
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8096, shuffle=False)

if EVAL_TYPE == "mlp": 
  pred_model = MLP(input_dim=256, dim_hidden=64, num_layers=2, out_dims=1).double().to(device)
elif EVAL_TYPE == "linear":
  pred_model = nn.Linear(train_embeddings.shape[1], 1, dtype=torch.float64).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(pred_model.parameters(), lr=1e-3)

epoch_losses = []
epochs = 100
running_loss = torch.zeros(1, device=device)

for epoch in range(epochs):
  pred_model.train()
  running_loss.zero_()  # Reset running loss for the epoch

  for batch in train_loader:
    optimizer.zero_grad()
    embeddings, values = batch
    # Forward pass
    y_pred = pred_model(embeddings.to(device))
    # Compute the loss
    loss = criterion(y_pred, values.to(device))
    # Backward pass
    loss.backward()
    # Update the parameters
    optimizer.step()
    # Append the loss to the list
    # losses.append(loss.item())

    running_loss += loss.detach() * embeddings.size(0)  # Multiply by batch size to get total loss for the batch

  epoch_loss = (running_loss / len(train_loader.dataset)).item()
  epoch_losses.append(epoch_loss)

  if (epoch) % 10 == 0:
    print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.4f}")

In [ ]:
# Plot training curves for the fine-tuned model
plt.plot(list(range(len(epoch_losses))), epoch_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves (Fine-tuned)')
plt.legend()
plt.show()

In [ ]:
# Run test
preds = []
actuals = []

with torch.no_grad():
  pred_model.eval()

  for batch in test_loader:
    embeddings, values = batch
    y_pred_test = pred_model(embeddings.to(device))
    preds.extend(y_pred_test.cpu().detach().numpy())
    actuals.extend(values.float().cpu().detach().numpy())

# Print test loss
print(f'Test loss: {criterion(torch.tensor(preds), torch.tensor(actuals)).item()}')

# Print test RMSE
test_rmse = torch.sqrt(F.mse_loss(torch.tensor(preds), torch.tensor(actuals))).item()
print(f'Test RMSE: {test_rmse/10} °C')

In [ ]:
# Plot predicted vs actual temperatures for the test set
plt.figure(figsize=(12, 6))
plt.scatter(actuals, preds, alpha=0.5)
plt.plot(actuals, actuals, color='red', linestyle='--')  # Line y=x for reference
plt.xlabel('Actual Temperature (normalized)')
plt.ylabel('Predicted Temperature (normalized)')
plt.title('Predicted vs Actual Temperatures for Test Set')

In [ ]:
# Select random station from the test set
random_station = random.choice(ids[(splits != train_split).numpy()].to_numpy())
station_name = ghcn_qc_temp_df[ghcn_qc_temp_df['ID'] == random_station]['Name'].values[0]

# random_station = random.choice(ids.unique())
station_data = temp_data[(ids == random_station).to_numpy(), :]
station_dates = dates[(ids == random_station).to_numpy()]
station_splits = splits[(ids == random_station).to_numpy()]
station_coords = station_data[:, :3].detach().clone()
station_embeddings = spatiotemporal_enc(station_coords.double().to(device)).detach()
station_preds = pred_model(station_embeddings.double().to(device)).cpu().detach().numpy()

plt.figure(figsize=(12, 6))
plt.scatter(station_dates, station_data[:, 3]/10, marker='o', linestyle='-', label=f'All data from station')
plt.scatter(station_dates, station_preds/10, marker='o', color="red", linestyle='-', label=f'Predicted data from station')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Over Time for Station ID {random_station}, lon/lat: {station_coords[0, 0].item():.4f}, {station_coords[0, 1].item():.4f}, Name: {station_name}')

In [ ]:
# Load the GTLoc model
import gtloc
gtloc_ckpt = "/home/leca5365/Documents/gtloc/ckpts/gtloc.pt"
gtl_model = gtloc.GTLoc(
        hidden_dim= 768,
        embedding_dim= 512,
        queue_size=4096,
        time_sigma=[2**0, 2**4, 2**8],
        loc_sigma= [2**0, 2**4, 2**8],
        freeze_backbone=True,
        galleries='data_dist',
        time_dropout=0.1,
)
state_dict = torch.load(gtloc_ckpt, map_location='cpu', weights_only=True)
gtl_model.load_state_dict(state_dict, strict=False)

gtl_model = gtl_model.to(device)
gtl_loc_encoder = gtl_model.location_encoder
gtl_time_encoder = gtl_model.time_encoder

# Put times in the correct format
num_dates = len(dates)
month_days = torch.zeros(num_dates, 5)
month_days[:, 0] = torch.from_numpy(dates.dt.month.values).float()
month_days[:, 1] = torch.from_numpy(dates.dt.day.values).float()

# Put locations into the correct format, from lon_lat to lat_lon
lat_lon = temp_data[:, :2]
lat_lon[:, 0], lat_lon[:, 1] = lat_lon[:, 1], lat_lon[:, 0]

with torch.no_grad():
    loc_emb = gtl_loc_encoder(lat_lon.to(device)).detach().cpu()
    time_emb = gtl_time_encoder(month_days.to(device)).detach().cpu()
    gtl_emb = torch.cat([loc_emb, time_emb], dim=1)

train_gtl_emb = gtl_emb[splits == train_split, :]
test_gtl_emb = gtl_emb[splits != train_split, :]

print("Location embeddings shape:", loc_emb.shape)
print("Time embeddings shape:", time_emb.shape)
print("Combined embeddings shape:", gtl_emb.shape)
print("Train embeddings shape:", train_gtl_emb.shape)
print("Test embeddings shape:", test_gtl_emb.shape)

In [ ]:
# Create a DataLoader for the train embeddings and values
train_dataset = torch.utils.data.TensorDataset(train_gtl_emb, train_sample[:, 3].detach().clone().unsqueeze(1))
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8096, shuffle=True)

# Test dataloader
test_dataset = torch.utils.data.TensorDataset(test_gtl_emb, test_sample[:, 3].detach().clone().unsqueeze(1))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8096, shuffle=False)

if EVAL_TYPE == "mlp": 
  pred_model = MLP(input_dim=train_gtl_emb.shape[1], dim_hidden=64, num_layers=2, out_dims=1).float().to(device)
elif EVAL_TYPE == "linear":
  pred_model = nn.Linear(train_gtl_emb.shape[1], 1, dtype=torch.float32).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(pred_model.parameters(), lr=1e-3)

epoch_losses = []
epochs = 100
running_loss = torch.zeros(1, device=device)

for epoch in range(epochs):
  pred_model.train()
  running_loss.zero_()  # Reset running loss for the epoch

  for batch in train_loader:
    optimizer.zero_grad()
    embeddings, values = batch
    # Forward pass
    y_pred = pred_model(embeddings.to(device).float())
    # Compute the loss
    loss = criterion(y_pred, values.to(device).float())
    # Backward pass
    loss.backward()
    # Update the parameters
    optimizer.step()
    # Append the loss to the list
    # losses.append(loss.item())

    running_loss += loss.detach() * embeddings.size(0)  # Multiply by batch size to get total loss for the batch

  epoch_loss = (running_loss / len(train_loader.dataset)).item()
  epoch_losses.append(epoch_loss)

  if (epoch) % 10 == 0:
    print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.4f}")

In [ ]:
# Run test
preds = []
actuals = []

with torch.no_grad():
  pred_model.eval()

  for batch in test_loader:
    embeddings, values = batch
    y_pred_test = pred_model(embeddings.float().to(device))
    preds.extend(y_pred_test.cpu().detach().numpy())
    actuals.extend(values.cpu().detach().numpy())

# Print test loss
print(f'Test loss: {criterion(torch.tensor(preds), torch.tensor(actuals)).item()}')

# Print test RMSE
test_rmse = torch.sqrt(F.mse_loss(torch.tensor(preds), torch.tensor(actuals))).item()
print(f'Test RMSE: {test_rmse/10} °C')

In [ ]:
# Plot predicted vs actual temperatures for the test set
plt.figure(figsize=(12, 6))
plt.scatter(actuals, preds, alpha=0.5)
plt.plot(actuals, actuals, color='red', linestyle='--')  # Line y=x for reference
plt.xlabel('Actual Temperature (normalized)')
plt.ylabel('Predicted Temperature (normalized)')
plt.title('Predicted vs Actual Temperatures for Test Set')

In [ ]:
# Create a DataLoader for the train embeddings and values
train_dataset = torch.utils.data.TensorDataset(train_sample[:, :3].detach().clone(), train_sample[:, 3].detach().clone().unsqueeze(1))
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8096, shuffle=True)

# Test dataloader
test_dataset = torch.utils.data.TensorDataset(test_sample[:, :3].detach().clone(), test_sample[:, 3].detach().clone().unsqueeze(1))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8096, shuffle=False)

print(EVAL_TYPE)
if EVAL_TYPE == "mlp": 
  simple_model = MLP(input_dim=3, dim_hidden=64, num_layers=2, out_dims=1).float().to(device)
elif EVAL_TYPE == "linear":
  simple_model = nn.Linear(train_sample[:, :3].shape[1], 1, dtype=torch.float32).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(simple_model.parameters(), lr=1e-3)

epoch_losses = []
epochs = 100
running_loss = torch.zeros(1, device=device)

for epoch in range(epochs):
  simple_model.train()
  running_loss.zero_()  # Reset running loss for the epoch

  for batch in train_loader:
    optimizer.zero_grad()
    coords, values = batch
    # Forward pass
    y_pred = simple_model(coords.to(device))
    # Compute the loss
    loss = criterion(y_pred, values.to(device))
    # Backward pass
    loss.backward()
    # Update the parameters
    optimizer.step()
    # Append the loss to the list
    # losses.append(loss.item())

    running_loss += loss.detach() * coords.size(0)  # Multiply by batch size to get total loss for the batch

  epoch_loss = (running_loss / len(train_loader.dataset)).item()
  epoch_losses.append(epoch_loss)

  if (epoch) % 10 == 0:
    print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.4f}")

In [ ]:
# Run test
preds = []
actuals = []

with torch.no_grad():
  simple_model.eval()

  for batch in test_loader:
    coords, values = batch
    y_pred_test = simple_model(coords.to(device))
    preds.extend(y_pred_test.cpu().detach().numpy())
    actuals.extend(values.to(device).cpu().detach().numpy())

# Print test loss
print(f'Test loss: {criterion(torch.tensor(preds), torch.tensor(actuals)).item()}')

# Print test RMSE
test_rmse = torch.sqrt(F.mse_loss(torch.tensor(preds), torch.tensor(actuals))).item()
print(f'Test RMSE: {test_rmse/10} °C')

In [ ]:
# Plot predicted vs actual temperatures for the test set
plt.figure(figsize=(12, 6))
plt.scatter(actuals, preds, alpha=0.5)
plt.plot(actuals, actuals, color='red', linestyle='--')  # Line y=x for reference
plt.xlabel('Actual Temperature (normalized)')
plt.ylabel('Predicted Temperature (normalized)')
plt.title('Predicted vs Actual Temperatures for Test Set')

In [ ]:
# Select random station from the test set
random_station = random.choice(ids[(splits != train_split).numpy()].to_numpy())
station_name = ghcn_qc_temp_df[ghcn_qc_temp_df['ID'] == random_station]['Name'].values[0]

# random_station = random.choice(ids.unique())
station_data = temp_data[(ids == random_station).to_numpy(), :]
station_dates = dates[(ids == random_station).to_numpy()]
station_splits = splits[(ids == random_station).to_numpy()]
station_coords = station_data[:, :3].detach().clone().to(device).float()
station_preds = simple_model(station_coords.float().to(device)).cpu().detach().numpy()

plt.figure(figsize=(12, 6))
plt.scatter(station_dates, station_data[:, 3]/10, marker='o', linestyle='-', label=f'All data from station')
plt.scatter(station_dates, station_preds/10, marker='o', color="red", linestyle='-', label=f'Predicted data from station')
plt.xlabel('Day of Year')
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Over Time for Station ID {random_station}, lon/lat: {station_coords[0, 0].item():.4f}, {station_coords[0, 1].item():.4f}, Name: {station_name}')

In [ ]:
# Plot training curves for the fine-tuned model
plt.plot(list(range(len(epoch_losses))), epoch_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves (Fine-tuned)')
plt.legend()
plt.show()

In [ ]:
# For testing purposes, assume input is a tensor of shape (batch_size, 3) representing (longitude, latitude, day_of_year) and month_day is a tensor of shape (batch_size, 2) representing (month, day). 
# The output will be a tensor of shape (batch_size, 1) representing the predicted temperature.

class TemperaturePredictor(nn.Module):
    def __init__(self, location_encoder, embed_dim=256, hidden_dim=64, num_layers=2, name="TempPredictor"):
        super(TemperaturePredictor, self).__init__()
        self.location_encoder = location_encoder
        self.name = name

        if location_encoder is None:
            self.fcn = MLP(input_dim=3, dim_hidden=hidden_dim, num_layers=num_layers, out_dims=1)
        else:
            self.fcn = MLP(input_dim=embed_dim, dim_hidden=hidden_dim, num_layers=num_layers, out_dims=1)

    def forward(self, coord_full):
        coord = coord_full[:, :3]  # Extract longitude and latitude
        month_day = coord_full[:, 3:]  # Extract month and day
        if self.location_encoder is None:
            return self.fcn(coord)
        if month_day is not None:
            loc_emb = self.location_encoder(coord, month_day)
        else:
            loc_emb = self.location_encoder(coord)
        temp_pred = self.fcn(loc_emb)
        return temp_pred

class GTLocCombinedEncoder(gtloc.GTLoc):
    def __init__(self, gtl_model):
        super(GTLocCombinedEncoder, self).__init__(
            hidden_dim=gtl_model.hidden_dim,
            embedding_dim=gtl_model.embedding_dim,
            queue_size=gtl_model.queue_size,
            time_sigma=gtl_model.time_sigma,
            loc_sigma=gtl_model.loc_sigma,
            freeze_backbone=gtl_model.freeze_backbone,
            galleries=gtl_model.galleries,
            time_dropout=gtl_model.time_dropout
        )
        self.location_encoder = gtl_model.location_encoder
        self.time_encoder = gtl_model.time_encoder

    def forward(self, coord, month_day):
        # Assumes coords are in [lon, lat] format and month_day is in [month, day] format
        # Swap the order of coordinates to [lat, lon] for the location encoder
        latlon = coord.clone()
        latlon[:, 0], latlon[:, 1] = latlon[:, 1], latlon[:, 0]
        loc_emb = self.location_encoder(latlon)
        time_emb = self.time_encoder(month_day)
        combined_emb = torch.cat([loc_emb, time_emb], dim=1)
        return combined_emb

In [ ]:
def _init_tp_models():
    satclip_tp = TemperaturePredictor(location_encoder=spatiotemporal_enc, embed_dim=256, hidden_dim=64, num_layers=2, name="SatClip_TempPredictor").double().to(device)
    gtloc_tp = TemperaturePredictor(location_encoder=GTLocCombinedEncoder(gtl_model), embed_dim=gtl_emb.shape[1], hidden_dim=64, num_layers=2, name="GTLoc_TempPredictor").float().to(device)
    mlp_tp = TemperaturePredictor(location_encoder=None, embed_dim=3, hidden_dim=64, num_layers=2, name="MLP_TempPredictor").float().to(device)
    return satclip_tp, gtloc_tp, mlp_tp

In [ ]:
# # Run spatial CV evals
# seeds = [42, 123, 456, 789, 101112]
# delta_degrees = [1, 8, 16, 32]

# logs = {} # {experiment_id: {seed, delta_degree, train_losses, test_loss, test_rmse, model}}

# for seed in tqdm(seeds, desc="Seed Loop"):
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)

#     for delta_degree in tqdm(delta_degrees, desc=f"Delta Degree Loop for Seed {seed}", leave=False):
#         # Create the spatial checkerboard splits for the GHCN dataset
#         delta_degree = 16
#         splits = checkerboard_splits(temp_data[:, :2], torch.tensor([delta_degree, delta_degree])).flatten()

#         # Setup initial datasets and dataloaders
#         train_split = 0

#         train_temp = temp_data[splits == train_split, :]
#         test_temp = temp_data[splits != train_split, :]

#         # Put times in the correct format
#         num_dates = len(dates)
#         month_days = torch.zeros(num_dates, 5)
#         month_days[:, 0] = torch.from_numpy(dates.dt.month.values).float()
#         month_days[:, 1] = torch.from_numpy(dates.dt.day.values).float()

#         train = torch.cat([train_temp[:,:3], month_days[splits == train_split, :], train_temp[:, 3].unsqueeze(1)], dim=1)
#         test = torch.cat([test_temp[:,:3], month_days[splits != train_split, :], test_temp[:, 3].unsqueeze(1)], dim=1)

#         train_dataset = torch.utils.data.TensorDataset(train[:, :3].detach().clone(), train[:, 3].detach().clone().unsqueeze(1))
#         train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8096, shuffle=True)

#         # Test dataloader
#         test_dataset = torch.utils.data.TensorDataset(test[:, :3].detach().clone(), test[:, 3].detach().clone().unsqueeze(1))
#         test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8096, shuffle=False)

#         # Run training loops
#         for model in _init_tp_models():
#             criterion = nn.MSELoss()
#             optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

#             epoch_losses = []
#             epochs = 100
#             running_loss = torch.zeros(1, device=device)

#             for epoch in tqdm(range(epochs), desc=f"Training {model.name} for Seed {seed}, Delta Degree {delta_degree}", leave=False):
#                 model.train()
#                 running_loss.zero_()  # Reset running loss for the epoch

#                 for batch in train_loader:
#                     optimizer.zero_grad()
#                     coords, values = batch
#                     # Forward pass
#                     y_pred = model(coords.to(device))
#                     # Compute the loss
#                     loss = criterion(y_pred, values.to(device))
#                     # Backward pass
#                     loss.backward()
#                     # Update the parameters
#                     optimizer.step()
#                     # Append the loss to the list
#                     # losses.append(loss.item())

#                     running_loss += loss.detach() * coords.size(0)  # Multiply by batch size to get total loss for the batch

#                     epoch_loss = (running_loss / len(train_loader.dataset)).item()
#                     epoch_losses.append(epoch_loss)

#                     if (epoch) % 10 == 0:
#                         print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.4f}")

#                 # Evaluate on the test set
#                 preds = []
#                 actuals = []
#                 with torch.no_grad():
#                     model.eval()

#                     for batch in test_loader:
#                         coords, values = batch
#                         y_pred_test = model(coords.to(device))
#                         preds.extend(y_pred_test.cpu().detach().numpy())
#                         actuals.extend(values.to(device).cpu().detach().numpy())

#                 # Print test loss
#                 test_loss = criterion(torch.tensor(preds), torch.tensor(actuals)).item()
#                 test_rmse = torch.sqrt(F.mse_loss(torch.tensor(preds), torch.tensor(actuals))).item()
#                 print(f'Test loss: {test_loss}, Test RMSE: {test_rmse/10} °C')

#                 logs[f"{model.name}_seed{seed}_delta{delta_degree}"] = {
#                     "seed": seed,
#                     "delta_degree": delta_degree,
#                     "train_losses": epoch_losses,
#                     "test_loss": test_loss,
#                     "test_rmse": test_rmse,
#                     "model": model.state_dict() 
#                 }




In [ ]:
# Run temporal CV evals
# seeds = [42, 123, 456, 789, 101112]
# delta_days = [1, 7, 30, 90]

# for seed in seeds:
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)

#     # Create temporal splits for the GHCN dataset
#     delta_day = 30
#     # convert dates into days since the first date
#     days_since_first = torch.tensor((dates - dates.min()).dt.days.to_numpy())
#     splits_shuffled = temporal_splits(days_since_first, delta_day)